In [1]:
# Load master file 
import pandas as pd 
master = pd.read_parquet("D:\\instacart_market_analysis\\master_table.parquet")
master.head()
day_map = {0:"Sunday", 1:"Monday", 2:"Tuesday", 3:"Wednesday",
           4:"Thursday", 5:"Friday", 6:"Saturday"}

In [2]:
user_stats = (master.groupby("user_id")
              .agg(
                  total_orders = ("order_id","nunique"),
                  total_products_bought = ("product_id","nunique"),
                  avg_basket_size = ("basket_size","mean"),
                  overall_reorder_rate = ("order_reorder_rate","mean")
                 )
                 .reset_index())
user_stats["avg_basket_size"] = user_stats["avg_basket_size"].round(4)
user_stats["overall_reorder_rate"] = user_stats["overall_reorder_rate"].round(2)
print(user_stats.shape)
print(user_stats.head())

(206209, 5)
   user_id  total_orders  total_products_bought  avg_basket_size  \
0        1            10                     18           6.2542   
1        2            14                    102          16.1077   
2        3            12                     33           7.8864   
3        4             5                     17           4.5556   
4        5             4                     23          10.0270   

   overall_reorder_rate  
0                  0.70  
1                  0.48  
2                  0.63  
3                  0.05  
4                  0.38  


In [3]:
# average days between order 
days = (master[master["days_since_prior_order"] != -1]
        .groupby("user_id")["days_since_prior_order"]
        .mean()
        .round(4)
        .reset_index()
        .rename(columns = {"days_since_prior_order":"avg_days_between_orders"}))
user_stats = user_stats.merge(days,on = "user_id",how = "left")
print(user_stats.head())

   user_id  total_orders  total_products_bought  avg_basket_size  \
0        1            10                     18           6.2542   
1        2            14                    102          16.1077   
2        3            12                     33           7.8864   
3        4             5                     17           4.5556   
4        5             4                     23          10.0270   

   overall_reorder_rate  avg_days_between_orders  
0                  0.70                  20.2593  
1                  0.48                  15.9670  
2                  0.63                  11.4872  
3                  0.05                  15.3571  
4                  0.38                  14.5000  


In [4]:
# favorate days per user

fav_day = (master.groupby("user_id")["order_dow"]
           .agg(lambda x: x.mode()[0])
           .reset_index()
           .rename(columns={"order_dow": "favourite_day_num"}))

fav_day["favourite_day"] = fav_day["favourite_day_num"].map(day_map)

user_stats = user_stats.merge(fav_day, on="user_id", how="left")
print(user_stats.head())

   user_id  total_orders  total_products_bought  avg_basket_size  \
0        1            10                     18           6.2542   
1        2            14                    102          16.1077   
2        3            12                     33           7.8864   
3        4             5                     17           4.5556   
4        5             4                     23          10.0270   

   overall_reorder_rate  avg_days_between_orders  favourite_day_num  \
0                  0.70                  20.2593                  4   
1                  0.48                  15.9670                  2   
2                  0.63                  11.4872                  0   
3                  0.05                  15.3571                  4   
4                  0.38                  14.5000                  3   

  favourite_day  
0      Thursday  
1       Tuesday  
2        Sunday  
3      Thursday  
4     Wednesday  


In [5]:
# favourite time of day per user
fav_time = (master.groupby("user_id")["time_of_order"]
            .agg(lambda x: x.mode()[0])
            .reset_index()
            .rename(columns={"time_of_order": "favourite_time"}))

user_stats = user_stats.merge(fav_time, on="user_id", how="left")
print(user_stats.head())

   user_id  total_orders  total_products_bought  avg_basket_size  \
0        1            10                     18           6.2542   
1        2            14                    102          16.1077   
2        3            12                     33           7.8864   
3        4             5                     17           4.5556   
4        5             4                     23          10.0270   

   overall_reorder_rate  avg_days_between_orders  favourite_day_num  \
0                  0.70                  20.2593                  4   
1                  0.48                  15.9670                  2   
2                  0.63                  11.4872                  0   
3                  0.05                  15.3571                  4   
4                  0.38                  14.5000                  3   

  favourite_day favourite_time  
0      Thursday        Morning  
1       Tuesday        Morning  
2        Sunday      Afternoon  
3      Thursday      Afternoon  

In [6]:
# customer segment
def segment(row):
    if   row["total_orders"] >= 10 and row["overall_reorder_rate"] >= 0.6:
        return "Loyal Regular"
    elif row["total_orders"] >= 10 and row["overall_reorder_rate"] <  0.6:
        return "Explorer"
    else:
        return "Occasional"

user_stats["customer_segment"] = user_stats.apply(segment, axis=1)

print("\nSegment breakdown:")
print(user_stats["customer_segment"].value_counts())


Segment breakdown:
customer_segment
Occasional       104513
Explorer          54636
Loyal Regular     47060
Name: count, dtype: int64


In [7]:
# average basket size by customer segment
basket_by_segment = (user_stats.groupby("customer_segment")["avg_basket_size"]
                     .mean()
                     .round(2)
                     .reset_index()
                     .sort_values("avg_basket_size", ascending=False))

print(basket_by_segment)

  customer_segment  avg_basket_size
1    Loyal Regular            12.83
0         Explorer            11.89
2       Occasional            11.37


In [8]:
# average reorder rate by customer segment
reorder_by_segment = (user_stats.groupby("customer_segment")["overall_reorder_rate"]
                      .mean()
                      .round(4)
                      .reset_index())

reorder_by_segment["reorder_rate_pct"] = (reorder_by_segment["overall_reorder_rate"] * 100).round(2)
reorder_by_segment = reorder_by_segment.sort_values("overall_reorder_rate", ascending=False)
print(reorder_by_segment)

  customer_segment  overall_reorder_rate  reorder_rate_pct
1    Loyal Regular                0.7127             71.27
0         Explorer                0.4447             44.47
2       Occasional                0.2994             29.94


In [9]:
# average days between orders by customer segment
days_by_segment = (user_stats.groupby("customer_segment")["avg_days_between_orders"]
                   .mean()
                   .round(2)
                   .reset_index()
                   .sort_values("avg_days_between_orders", ascending=True))

print(days_by_segment)

  customer_segment  avg_days_between_orders
1    Loyal Regular                    10.35
0         Explorer                    13.65
2       Occasional                    18.73


In [10]:
# final column order for user profiles
user_stats = user_stats[["user_id", "total_orders", "total_products_bought",
                          "avg_basket_size", "overall_reorder_rate",
                          "avg_days_between_orders", "favourite_day",
                          "favourite_day_num", "favourite_time",
                          "customer_segment"]]

print("Final user profiles shape:", user_stats.shape)
print(user_stats.head(10))

Final user profiles shape: (206209, 10)
   user_id  total_orders  total_products_bought  avg_basket_size  \
0        1            10                     18           6.2542   
1        2            14                    102          16.1077   
2        3            12                     33           7.8864   
3        4             5                     17           4.5556   
4        5             4                     23          10.0270   
5        6             3                     12           5.2857   
6        7            20                     68          13.5049   
7        8             3                     36          17.0408   
8        9             3                     58          29.5526   
9       10             5                     94          34.7622   

   overall_reorder_rate  avg_days_between_orders favourite_day  \
0                  0.70                20.259300      Thursday   
1                  0.48                15.967000       Tuesday   
2            

In [11]:
# save user profiles
user_stats.to_csv("D:\\instacart_market_analysis\\user_profiles.csv", index=False)
print("User profiles saved — shape:", user_stats.shape)

User profiles saved — shape: (206209, 10)
